# 11 · Aggregate anchor — a quarterly model at the level the business is held to, and the range `blend .. upside`

The monthly, per-series forecast of notebook 09 is the right *operational* grain, but it is not the number the
business signs off: that is the **quarter total per product family**. Two facts from the field motivate this
notebook:

1. **Aggregation removes noise but not level shifts.** Summing hundreds of monthly series forecasts is accurate in
   flat quarters and systematically *low* in surge quarters — a tree model cannot predict above its training range,
   and the misses do not cancel because they all point the same way.
2. **A pooled quarterly model is a better judge of the quarter total.** One LightGBM over the *series × fiscal
   quarter* panel of a family (cluster labels ignored), summed bottom-up, lands within ~10 % of the quarter total in
   most quarters. Its **upper quantile** is right in surge quarters and over-forecasts flat ones; its **central
   estimate** is right in flat quarters and misses surges. **Summing per-series medians (`q50`) under-forecasts a
   lumpy total** — use `q75` or the L2 mean for aggregates.

The resolution is not to pick one: publish a **range**. `blend` = the bottom-up monthly forecast (the number we are
held to, no scaling); `upside` = `blend` scaled so the family quarter total equals the quarterly model's upper
quantile (the number if the current surge continues); `plateau` = the same against the central estimate — kept as
a technical column that answers *"what if the surge does not continue?"*. The scaling is applied to the validation
folds exactly as to the served quarter, so the range has a measured track record before anyone sees it.

| | |
|---|---|
| **Reads** | `<scenario>_clustered` (08), `<scenario>_quantile_forecasts` + `_quantile_config` (09) |
| **Writes** | `<scenario>_anchor_backtest` (every output × family × quarter), `<scenario>_anchor_quarterly_views` (blend / upside / plateau vs actual per validation fold), `<scenario>_reconciled_forecasts` (served series-months with the three views) |
| **Aggregate level** | `product_family` (set `GROUP_COL` to any column constant per series) |

## 1. Imports and configuration

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

warnings.filterwarnings("ignore")

SCENARIO = "durable_goods_monthly"
UID, DATE_VAR, TARGET, QTY_COL = "unique_id", "ds", "y", "quantity"
GROUP_COL = "product_family"                 # the aggregate the business is held to
CATS = ["product_group", "region", "channel"]
ANCHOR_GROUPS = None                         # None = every group; or a list of group names
N_BACKTEST_Q = 5                             # complete quarters backtested leak-free
QUANTILES = [0.10, 0.25, 0.50, 0.70, 0.75, 0.90]
VIEWS = {"upside": "Q-LGBM raw q75", "plateau": "Q-LGBM ratio mean"}   # or 'backtest' = best output per group
BUSINESS_ESTIMATE_M = {}                     # optional, e.g. {"HVAC Units": (27.0, 30.0)} in millions for the served quarter
RANDOM_STATE = 42
LGBM_Q = {"objective": "quantile", "boosting_type": "gbdt", "random_state": RANDOM_STATE, "n_jobs": -1, "verbose": -1,
          "n_estimators": 300, "learning_rate": 0.03, "num_leaves": 6, "max_depth": 4, "min_child_samples": 15,
          "reg_alpha": 1.0, "reg_lambda": 5.0, "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8}
FEATS = ["lag1", "lag2", "lag3", "lag4", "ma4", "qoq", "yoy", "qty_lag1", "price_lag1", "fq"] + CATS

LAKEHOUSE_NAME, LOCAL_FOLDER = "ts_forecasting", SCENARIO
root_path = Path.cwd().parent.parent
local_dir = root_path / "data" / LOCAL_FOLDER
CLUSTERED_TABLE, FORECAST_TABLE, CONFIG_TABLE = f"{SCENARIO}_clustered", f"{SCENARIO}_quantile_forecasts", f"{SCENARIO}_quantile_config"
OUT_BACKTEST, OUT_VIEWS, OUT_RECON = f"{SCENARIO}_anchor_backtest", f"{SCENARIO}_anchor_quarterly_views", f"{SCENARIO}_reconciled_forecasts"


def load_table(name):
    try:
        df = spark.table(f"{LAKEHOUSE_NAME}.{name}").toPandas()
        print(f"✅ Loaded {LAKEHOUSE_NAME}.{name} ({len(df):,} rows)")
    except Exception as e:
        df = pd.read_parquet(local_dir / f"{name}.parquet")
        print(f"ℹ️ Spark unavailable ({type(e).__name__}); loaded {local_dir / (name + '.parquet')} ({len(df):,} rows)")
    return df


def save_table(df, name):
    try:
        spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{LAKEHOUSE_NAME}.{name}")
        print(f"✅ Saved {len(df):,} rows to {LAKEHOUSE_NAME}.{name}")
    except Exception as e:
        local_dir.mkdir(parents=True, exist_ok=True)
        df.to_parquet(local_dir / f"{name}.parquet", index=False)
        print(f"ℹ️ Spark unavailable ({type(e).__name__}); saved {len(df):,} rows to {local_dir / (name + '.parquet')}")


print(f"✅ Config | aggregate level {GROUP_COL} | views {VIEWS} | {N_BACKTEST_Q} backtest quarters")

## 2. Load the panel, the monthly forecasts and the fiscal calendar

In [ ]:
panel = load_table(CLUSTERED_TABLE)
panel[DATE_VAR] = pd.to_datetime(panel[DATE_VAR])
fc = load_table(FORECAST_TABLE)
fc[DATE_VAR] = pd.to_datetime(fc[DATE_VAR])
cfg = json.loads(load_table(CONFIG_TABLE)["json"].iloc[0])
FY_START_MONTH = int(cfg.get("fy_start_month", 10))
DATA_MAX = panel[DATE_VAR].max()
GROUP_OF = panel.groupby(UID)[GROUP_COL].first()
GROUPS = ANCHOR_GROUPS or sorted(panel[GROUP_COL].dropna().unique())


def fiscal_parts(ts):
    fm = ((ts.month - FY_START_MONTH) % 12) + 1
    return ts.year + (1 if ts.month >= FY_START_MONTH else 0), (fm - 1) // 3 + 1, fm


def qindex(ts):
    fy, fq, _ = fiscal_parts(ts)
    return fy * 4 + (fq - 1)


def qlabel(qi):
    return f"FY{(qi // 4) % 100:02d}-Q{qi % 4 + 1}"


def qmonths(qi):
    fy, fq = qi // 4, qi % 4 + 1
    m0 = (FY_START_MONTH + 3 * (fq - 1) - 1) % 12 + 1
    y0 = fy - 1 if m0 >= FY_START_MONTH else fy
    return [pd.Timestamp(year=y0, month=m0, day=1) + pd.DateOffset(months=i) for i in range(3)]


panel["qi"] = panel[DATE_VAR].map(qindex)
_months_per_q = panel.groupby("qi")[DATE_VAR].nunique()
COMPLETE_Q = sorted(_months_per_q[_months_per_q == 3].index)
SERVED_Q = qindex(pd.to_datetime(fc.loc[fc["fold"] == "final", DATE_VAR].min()))
BACKTEST_Q = COMPLETE_Q[-N_BACKTEST_Q:]
print(f"data through {DATA_MAX.date()} | complete quarters {qlabel(COMPLETE_Q[0])} .. {qlabel(COMPLETE_Q[-1])} | "
      f"backtest {[qlabel(q) for q in BACKTEST_Q]} | served {qlabel(SERVED_Q)}")
print(f"groups: {GROUPS}")

## 3. The quarterly panel and the two targets

One row per *series × fiscal quarter*. Features are quarterly lags (`lag1..lag4`), their mean, QoQ and YoY ratios
(clipped so a near-zero base cannot explode them), lagged quantity and price, the fiscal quarter position and the
identity categoricals. Two ways to fit the same model:

- **raw** — the target is the quarter revenue; large series dominate the loss (which is what you want when the
  question is the *total*).
- **ratio** — the target is `y / lvl` with `lvl` = mean |lag1..lag4| (a floored level), so every series is on a
  comparable scale and growth multipliers transfer across series; predictions are scaled back by `lvl`. Rows are
  weighted by `lvl / mean(lvl)` so dollars still count — and **the weights are normalised to mean 1**, because
  LightGBM's quantile objective misbehaves with raw-dollar sample weights.

In [ ]:
def quarterly_panel(d, extra_q=None):
    q = (d.groupby([UID, "qi"], observed=True)
           .agg(y=(TARGET, "sum"), qty=(QTY_COL, "sum"), **{c: (c, "last") for c in CATS}).reset_index())
    q = q[q["qi"].isin(COMPLETE_Q)]
    quarters = sorted(set(q["qi"]) | ({extra_q} if extra_q is not None else set()))
    idx = pd.MultiIndex.from_product([q[UID].unique(), quarters], names=[UID, "qi"])
    q = q.set_index([UID, "qi"]).reindex(idx).reset_index()
    hist = (q["qi"] != extra_q) if extra_q is not None else pd.Series(True, index=q.index)
    q.loc[hist, ["y", "qty"]] = q.loc[hist, ["y", "qty"]].fillna(0.0)
    for c in CATS:
        q[c] = q.groupby(UID)[c].transform(lambda s: s.ffill().bfill())
    q = q.sort_values([UID, "qi"]).reset_index(drop=True)
    g = q.groupby(UID)
    for k in (1, 2, 3, 4, 5):
        q[f"lag{k}"] = g["y"].shift(k)
    q["ma4"] = q[["lag1", "lag2", "lag3", "lag4"]].mean(axis=1)
    q["qoq"] = (q["lag1"] / q["lag2"].replace(0, np.nan)).clip(0.2, 5.0)
    q["yoy"] = (q["lag1"] / q["lag5"].replace(0, np.nan)).clip(0.2, 5.0)
    q["qty_lag1"] = g["qty"].shift(1)
    q["price_lag1"] = q["lag1"] / q["qty_lag1"].replace(0, np.nan)
    q["fq"] = q["qi"] % 4 + 1
    lvl = q[["lag1", "lag2", "lag3", "lag4"]].abs().mean(axis=1)
    floor = float(np.nanpercentile(lvl[lvl > 0], 5)) if (lvl > 0).any() else 1.0
    q["lvl"] = np.where(lvl.notna(), np.maximum(lvl, floor), np.nan)
    return q


def fit_predict(train, test, target="raw"):
    tr = train.dropna(subset=["y"]).copy()
    if target == "ratio":
        tr = tr[tr["lvl"] > 0]
        yt, w = (tr["y"] / tr["lvl"]).clip(-2.0, 6.0), tr["lvl"] / tr["lvl"].mean()
    else:
        yt, w = tr["y"], None
    test = test.copy()
    for c in CATS:
        cat = pd.CategoricalDtype(sorted(set(train[c].dropna().astype(str)) | set(test[c].dropna().astype(str))))
        tr[c], test[c] = tr[c].astype(str).astype(cat), test[c].astype(str).astype(cat)
    X, Xt = tr[FEATS], test[FEATS]
    scale = test["lvl"].to_numpy(float) if target == "ratio" else 1.0
    out = pd.DataFrame(index=test.index)
    for qq in QUANTILES:
        out[f"q{int(qq * 100)}"] = lgb.LGBMRegressor(**{**LGBM_Q, "alpha": qq}).fit(X, yt, sample_weight=w, categorical_feature=CATS).predict(Xt) * scale
    out["mean"] = lgb.LGBMRegressor(**{**LGBM_Q, "objective": "l2"}).fit(X, yt, sample_weight=w, categorical_feature=CATS).predict(Xt) * scale
    if target == "ratio":
        miss = ~np.isfinite(scale)
        for c in out.columns:
            out.loc[miss, c] = test.loc[miss, "lag1"].to_numpy()
    return out


def outputs(q, train, test, tq):
    """Every candidate for the quarter total: simple totals + both model families, summed bottom-up."""
    tot = q[q["qi"] < tq].groupby("qi")["y"].sum()
    fcs = {"TOTAL last quarter": tot.get(tq - 1, np.nan), "TOTAL snaive": tot.get(tq - 4, np.nan),
           "TOTAL yoy-growth naive": (tot.get(tq - 4, np.nan) * tot.get(tq - 1, np.nan) / tot.get(tq - 5, np.nan)
                                      if (tq - 5) in tot.index and tot.get(tq - 5, 0) > 0 else np.nan)}
    band = {}
    for tgt in ("raw", "ratio"):
        p = fit_predict(train, test, tgt)
        for c in ("q50", "q70", "q75", "q90", "mean"):
            fcs[f"Q-LGBM {tgt} {c}"] = float(p[c].sum())
        band[tgt] = (float(p["q10"].sum()), float(p["q90"].sum()))
    return fcs, band


print("✅ quarterly engine ready")

## 4. Leak-free backtest of every output on the quarter total

In [ ]:
rows = []
QP = {}
for g in GROUPS:
    d = panel[panel[GROUP_COL] == g]
    q = quarterly_panel(d)
    QP[g] = q
    for tq in BACKTEST_Q:
        train, test = q[(q["qi"] < tq) & q["lag1"].notna()], q[q["qi"] == tq]
        if len(train) < 30 or test.empty:
            continue
        actual = float(test["y"].sum())
        fcs, band = outputs(q, train, test, tq)
        for k, v in fcs.items():
            if v is not None and np.isfinite(v):
                rows.append({"group": g, "quarter": qlabel(tq), "qi": tq, "method": k, "actual_M": actual / 1e6, "forecast_M": v / 1e6,
                             "error_pct": (v - actual) / abs(actual) * 100.0, "ape_pct": abs(v - actual) / abs(actual) * 100.0})
    print(f"  {g}: backtested {len(BACKTEST_Q)} quarters")
backtest = pd.DataFrame(rows)

summary = (backtest.groupby(["group", "method"])
           .agg(mean_ape=("ape_pct", "mean"), max_ape=("ape_pct", "max"), mean_signed=("error_pct", "mean"),
                within_10=("ape_pct", lambda s: int((s <= 10).sum())))
           .reset_index())
BEST = summary.sort_values(["group", "mean_ape"]).groupby("group").first()["method"]
print("\nMean APE % of the quarter total per output (lower is better); mean signed error shows the direction:")
pivot = summary.pivot(index="method", columns="group", values="mean_ape").round(1)
print(pivot.loc[sorted(pivot.index, key=lambda m: pivot.loc[m].mean())].to_string())
print("\nbacktest-best output per group:\n" + BEST.to_string())

### The lesson in one table: medians under-forecast lumpy totals; the upper quantile and the mean do not

Signed error of the summed per-series outputs. For the families dominated by project deals the sum of `q50`
sits well below the actual total in every quarter — the median of a right-skewed distribution is below its mean,
and sums add means. The `q75` / `mean` outputs are close to unbiased on the total.

In [ ]:
lesson = (summary[summary["method"].isin(["Q-LGBM raw q50", "Q-LGBM raw q75", "Q-LGBM raw mean", "Q-LGBM ratio q50", "Q-LGBM ratio mean", "TOTAL snaive"])]
          .pivot(index="group", columns="method", values="mean_signed").round(1))
print("mean signed error % of the quarter total (negative = under-forecast):")
print(lesson.to_string())

fig, ax = plt.subplots(figsize=(10, 4.2))
hm = summary.pivot(index="group", columns="method", values="mean_ape")
hm = hm[sorted(hm.columns, key=lambda m: hm[m].mean())]
im = ax.imshow(hm.to_numpy(float), cmap="RdYlGn_r", vmin=0, vmax=max(30, np.nanpercentile(hm.to_numpy(float), 90)))
ax.set_xticks(range(len(hm.columns)))
ax.set_xticklabels(hm.columns, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(hm.index)))
ax.set_yticklabels(hm.index, fontsize=9)
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        ax.text(j, i, f"{hm.iat[i, j]:.0f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, label="mean APE % of the quarter total")
ax.set_title(f"Quarterly anchor backtest - {N_BACKTEST_Q} leak-free quarters per {GROUP_COL}", loc="left")
fig.tight_layout()
plt.show()

## 5. Served quarter: full-quarter forecast + booked months (nowcast)

The served quarter is partly booked. The anchor for the quarter total is `booked actual + full-quarter forecast ×
(1 − typical share of the booked month positions)`, where the share comes from the last four complete quarters.
Each view (`upside`, `plateau`) picks its output by name, or `'backtest'` = the best output of the backtest.

In [ ]:
ANCHOR = {}
served_months = qmonths(SERVED_Q)
known_months = [m for m in served_months if m <= DATA_MAX]
pos = [served_months.index(m) for m in known_months]
for g in GROUPS:
    d = panel[panel[GROUP_COL] == g]
    qs = quarterly_panel(d, extra_q=SERVED_Q)
    train, test = qs[(qs["qi"] < SERVED_Q) & qs["lag1"].notna()], qs[qs["qi"] == SERVED_Q]
    known_actual = float(d[d[DATE_VAR].isin(known_months)][TARGET].sum())
    shares = []
    for qi in COMPLETE_Q[-4:]:
        mm = qmonths(qi)
        tot = d[d[DATE_VAR].isin(mm)][TARGET].sum()
        if tot > 0 and pos:
            shares.append(d[d[DATE_VAR].isin([mm[i] for i in pos])][TARGET].sum() / tot)
    share_known = float(np.mean(shares)) if shares else 0.0
    fcs, band = outputs(qs, train, test, SERVED_Q)
    served_total = {k: known_actual + v * (1.0 - share_known) for k, v in fcs.items() if v is not None and np.isfinite(v)}
    views = {}
    for view, want in VIEWS.items():
        m = BEST[g] if want == "backtest" else want
        if m not in served_total:
            m = BEST[g]
        bt = summary[(summary["group"] == g) & (summary["method"] == m)].iloc[0]
        views[view] = {"method": m, "full_quarter": fcs[m], "served_total": served_total[m], "backtest_mean_ape": float(bt["mean_ape"])}
    ANCHOR[g] = {"quarter": qlabel(SERVED_Q), "known_months": [m.strftime("%Y-%m") for m in known_months], "known_actual": known_actual,
                 "share_known": share_known, "views": views, "backtest_best": BEST[g],
                 "band_q10_q90_M": tuple((known_actual + b * (1 - share_known)) / 1e6 for b in band["raw"])}
    print(f"{g:24s} | booked {ANCHOR[g]['known_months']} = {known_actual/1e6:.2f} M ({share_known:.0%} typical share) | "
          + " | ".join(f"{v}={views[v]['served_total']/1e6:.2f} M via {views[v]['method']!r} (backtest APE {views[v]['backtest_mean_ape']:.1f})" for v in views)
          + f" | q10-q90 {ANCHOR[g]['band_q10_q90_M'][0]:.1f}-{ANCHOR[g]['band_q10_q90_M'][1]:.1f} M")
    if g in BUSINESS_ESTIMATE_M:
        lo, hi = BUSINESS_ESTIMATE_M[g]
        print(f"{'':24s}   business estimate {lo}-{hi} M -> " + ", ".join(
            f"{v} {'WITHIN' if lo <= views[v]['served_total']/1e6 <= hi else 'OUTSIDE'}" for v in views))

## 6. Reconcile the monthly bottom-up to the anchor — the range `blend .. upside`

`blend` is the notebook-09 committed forecast summed over the family (tuned clusters) plus the seasonal naive for
every other series — no scaling. For each family and view **one factor** scales the not-yet-booked months so the
family quarter total equals the view's anchor; booked months keep their actuals in every view. The same factor logic
is applied to each validation fold (anchor = that fold's backtest forecast), so the range is scored before it is
published.

In [ ]:
fc["group"] = fc[UID].map(GROUP_OF)
actual_by = panel.set_index([UID, DATE_VAR])[TARGET]
last_by = panel.sort_values(DATE_VAR).groupby(UID)[TARGET].last()


def naive_for(uids, months):
    return sum(max(float(actual_by.get((u, m - pd.DateOffset(months=12)), last_by.get(u, 0.0))), 0.0) for u in uids for m in months)


# ---- validation folds: bottom-up per group vs actual, scaled to the fold's backtest anchor ----
fold_rows = []
for fold, f in fc[fc["fold"] != "final"].groupby("fold"):
    months = sorted(f[DATE_VAR].unique())
    label = f["quarter"].iloc[0]
    for g in GROUPS:
        sub = f[f["group"] == g]
        tuned_ids = set(sub[UID])
        other_ids = [u for u, gg in GROUP_OF.items() if gg == g and u not in tuned_ids]
        bottom = float(sub["yhat_committed"].sum()) + naive_for(other_ids, months)
        actual = float(panel[(panel[GROUP_COL] == g) & panel[DATE_VAR].isin(months)][TARGET].sum())
        row = {"group": g, "fold": fold, "quarter": label, "actual_M": actual / 1e6, "blend_M": bottom / 1e6,
               "blend_ape": abs(bottom - actual) / abs(actual) * 100 if actual else np.nan}
        for view, v in ANCHOR[g]["views"].items():
            bt = backtest[(backtest["group"] == g) & (backtest["quarter"] == label) & (backtest["method"] == v["method"])]
            anchor = float(bt["forecast_M"].iloc[0]) * 1e6 if len(bt) else np.nan
            k = anchor / bottom if np.isfinite(anchor) and bottom > 0 else 1.0
            row[f"{view}_M"], row[f"{view}_factor"] = bottom * k / 1e6, k
            row[f"{view}_ape"] = abs(bottom * k - actual) / abs(actual) * 100 if actual else np.nan
        fold_rows.append(row)
views_df = pd.DataFrame(fold_rows).sort_values(["group", "fold"])
print("Validation folds - quarter total per group (M): blend .. upside, plateau = technical")
print(views_df[["group", "quarter", "actual_M", "blend_M", "blend_ape", "upside_M", "upside_ape", "upside_factor", "plateau_M", "plateau_ape"]].round(2).to_string(index=False))
print("\nmean APE % over the folds:")
print(views_df.groupby("group")[["blend_ape", "upside_ape", "plateau_ape"]].mean().round(1).to_string())
print(f"\nportfolio (all groups) mean APE: blend {views_df.groupby('fold')[['blend_M','actual_M']].sum().pipe(lambda t: (abs(t.blend_M-t.actual_M)/t.actual_M*100).mean()):.1f} | "
      f"upside {views_df.groupby('fold')[['upside_M','actual_M']].sum().pipe(lambda t: (abs(t.upside_M-t.actual_M)/t.actual_M*100).mean()):.1f} | "
      f"plateau {views_df.groupby('fold')[['plateau_M','actual_M']].sum().pipe(lambda t: (abs(t.plateau_M-t.actual_M)/t.actual_M*100).mean()):.1f}")

In [ ]:
# ---- served quarter: scale the not-yet-booked months of every series in the group ----
served = fc[fc["fold"] == "final"].copy()
served["forecast_blend"] = served["yhat_committed"]
for view in VIEWS:
    served[f"forecast_{view}"] = served["forecast_blend"]
    served[f"{view}_factor"], served[f"{view}_anchor"] = 1.0, ""
for g, a in ANCHOR.items():
    m = (served["group"] == g).to_numpy()
    fut = m & served["is_future"].to_numpy()
    known_sum = float(served.loc[m & ~served["is_future"].to_numpy(), "forecast_blend"].sum())
    bottom = float(served.loc[fut, "forecast_blend"].sum())
    for view, v in a["views"].items():
        target = v["served_total"] - known_sum
        if bottom <= 0 or not np.isfinite(target) or target <= 0:
            print(f"[{view}] {g}: cannot scale (bottom {bottom:,.0f}, target {target:,.0f}) -> {view} = blend")
            continue
        k = target / bottom
        served.loc[fut, f"forecast_{view}"] = served.loc[fut, "forecast_blend"] * k
        served.loc[m, f"{view}_factor"] = k
        served.loc[m, f"{view}_anchor"] = f"{v['method']} @ {a['quarter']}"
tab = (served.groupby("group").agg(blend_M=("forecast_blend", lambda s: s.sum() / 1e6), upside_M=("forecast_upside", lambda s: s.sum() / 1e6),
                                   plateau_M=("forecast_plateau", lambda s: s.sum() / 1e6), snaive_M=("snaive", lambda s: s.sum() / 1e6),
                                   upside_factor=("upside_factor", "max"), plateau_factor=("plateau_factor", "max")))
tab["upside_output"] = [ANCHOR[g]["views"]["upside"]["method"] for g in tab.index]
tab["plateau_output"] = [ANCHOR[g]["views"]["plateau"]["method"] for g in tab.index]
tab["range_lo_M"], tab["range_hi_M"] = tab[["blend_M", "upside_M"]].min(axis=1), tab[["blend_M", "upside_M"]].max(axis=1)
print(f"Served quarter {qlabel(SERVED_Q)} per {GROUP_COL} (M): published range = blend .. upside (shown as lo .. hi); plateau = technical")
print(tab.round(2).to_string())
print("\nNote: a factor below 1 means the quarterly anchor sits BELOW the bottom-up - typical when a rules layer or a"
      " strategic series lifted the monthly blend; the range then reads upside .. blend. The two numbers still bracket the quarter.")
print(f"\nportfolio: blend {served['forecast_blend'].sum()/1e6:.1f} M | upside {served['forecast_upside'].sum()/1e6:.1f} M | "
      f"plateau {served['forecast_plateau'].sum()/1e6:.1f} M | seasonal naive {served['snaive'].sum()/1e6:.1f} M")

## 7. Chart — quarter totals per family: actual, blend, upside, plateau across the folds and the served quarter

In [ ]:
qa = panel.groupby([GROUP_COL, "qi"])[TARGET].sum().div(1e6).reset_index()
fig, axes = plt.subplots(len(GROUPS), 1, figsize=(11, 2.9 * len(GROUPS)), sharex=False)
for ax, g in zip(np.atleast_1d(axes), GROUPS):
    h = qa[(qa[GROUP_COL] == g) & qa["qi"].isin(COMPLETE_Q[-10:])]
    ax.plot(h["qi"], h[TARGET], "k-o", ms=4, label="actual (complete quarters)")
    v = views_df[views_df["group"] == g]
    qi_v = [int(backtest.loc[backtest["quarter"] == q, "qi"].iloc[0]) for q in v["quarter"]]
    ax.plot(qi_v, v["blend_M"], "s", color="#1f77b4", ms=8, label="blend (bottom-up)")
    ax.plot(qi_v, v["upside_M"], "D", color="#d62728", ms=7, label="upside")
    ax.plot(qi_v, v["plateau_M"], "x", color="#7f7f7f", ms=8, label="plateau (technical)")
    ax.vlines(qi_v, v["blend_M"], v["upside_M"], color="#1f77b4", alpha=0.3, linewidth=6)
    a = ANCHOR[g]
    ax.plot([SERVED_Q], [tab.loc[g, "blend_M"]], "s", color="#1f77b4", ms=8, mfc="none")
    ax.plot([SERVED_Q], [tab.loc[g, "upside_M"]], "D", color="#d62728", ms=7, mfc="none")
    ax.plot([SERVED_Q], [tab.loc[g, "plateau_M"]], "x", color="#7f7f7f", ms=8)
    ax.vlines([SERVED_Q], tab.loc[g, "blend_M"], tab.loc[g, "upside_M"], color="#d62728", alpha=0.25, linewidth=6)
    ax.axvspan(SERVED_Q - 0.5, SERVED_Q + 0.5, color="orange", alpha=0.12)
    ticks = sorted(set(h["qi"]) | {SERVED_Q})
    ax.set_xticks(ticks)
    ax.set_xticklabels([qlabel(t) for t in ticks], rotation=45, fontsize=8)
    ax.set_title(f"{g} - quarter total (M); served {qlabel(SERVED_Q)} range {tab.loc[g, 'range_lo_M']:.1f} .. {tab.loc[g, 'range_hi_M']:.1f}", loc="left", fontsize=10)
    ax.set_ylabel("M")
np.atleast_1d(axes)[0].legend(fontsize=7, ncol=4)
fig.tight_layout()
plt.show()

## 8. Persist

In [ ]:
save_table(backtest, OUT_BACKTEST)
save_table(views_df, OUT_VIEWS)
keep = [UID, DATE_VAR, "group", "profile_cluster", "quarter", "is_future", "y", "forecast_blend", "forecast_upside", "forecast_plateau",
        "upside_factor", "plateau_factor", "upside_anchor", "plateau_anchor", "snaive", "cluster_method"]
save_table(served[[c for c in keep if c in served.columns]], OUT_RECON)

## ✅ CHECK POINT with the data scientist

- **Which output anchors which view is a decision, not a fit.** The backtest table shows, per family, when the
  upper quantile is right (surge quarters) and when the central estimate is (flat quarters). The published range
  `blend .. upside` carries both stories; `plateau` stays a technical column.
- **Read the fold table before the served table.** If `upside_ape` is not better than `blend_ape` on the recent
  folds for a family, do not reconcile that family — set `ANCHOR_GROUPS` to the ones where the anchor has a track
  record.
- **Sums of medians are not medians of sums.** Never anchor a lumpy total to `q50`.
- The business estimate check (`BUSINESS_ESTIMATE_M`) is the last step, not the first: the range is validated against
  history before it is compared with the plan.